# Intro

In this notebook we will quickly see how to set up our environment, and how to create a naive RAG that nontheless works.

First, we need to create a new folder for our project, initialize it, and add the dependencies:

```bash
mkdir llm-zoomcamp-2026-code
cd llm-zoomcamp-2026-code
uv init
uv add requests minsearch openai jupyter python-dotenv
```

Then, we need to set up our API keys; in my case, I used OpenAI. After obtaining my key, I created an `.env` file, made sure I added the file to my `.gitignore`, and then put my key in it.

```bash
OPENAI_API_KEY=sk-YOUR_KEY_HERE
```

## Intro to RAG

`dotenv` is the library we installed earlier (`python-dotenv`). It reads my `.env` file and loads the variables in it into the environment, so Python can access them.

`load_dotenv()` does the actual loading. After that line runs, my `OPENAI_API_KEY` is available as an environment variable, and the OpenAI client picks it up automatically. I don't have to write the key in the code.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
question = 'I just discovered the course. Can I join now?'

In [5]:
answer = llm(question)
print(answer)

Yes—usually you can join, but it depends on the course’s enrollment rules and whether it’s still open.

A few common possibilities:
- **Open enrollment:** You can join right away.
- **Cohort-based course:** You may need to wait for the next start date.
- **In-progress course:** You might still be able to join late, but you could miss earlier lessons or assignments.

If you want, I can help you write a quick message to the course team asking:
- whether late enrollment is allowed,
- how to catch up,
- and whether any spots are still available.


👆 As we can see, OpenAI doesn't really know if we can still join the course. Of course it doesn't! We need to enrich the users question with the relevant context, so that the API can generate for us richer and more precise answers.

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still open.


👆 Now that we have passed the relevant context, OpenAI can easily answer questions from our students!